In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model = "gpt-oss:20b",
    temperature=0
)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/home/aakash/Projects/AI/interview-practice/src/docs/04-Policy-Manual-Apr-2016-excl-SAICA.pdf")

documents = loader.load()

print(len(documents))
print(documents[0].page_content)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

chunks = text_splitter.split_documents(documents)

print(len(chunks))

for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i + 1} ---")
    print(chunk.page_content)
    print(chunk.metadata)

In [ ]:
from langchain_ollama import OllamaEmbeddings

embedding = OllamaEmbeddings(
    model= "nomic-embed-text:latest"
)

vector = embedding.embed_documents(
    [chunk.page_content for chunk in chunks]
)

print(vector[0])

In [ ]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    collection_name="policy_documents",
    persist_directory="./chroma_db"
)

print(vector_store)

In [ ]:
query = "What is the employee notice period?"

retriever = vector_store.as_retriever(
    search_kwargs={"k" : 3}
)


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpfull AI assistant answering question based on the company policy documents
Use only the following context to answer.
If answer not provided in the context.
say , I dont know.

Rules:
    Do not assume.
    Do not hallucinate.

Context:
    {context}
Question:
    {question}
""")

In [ ]:
def formatDoc(docs):
    return "\n\n".join(
        doc.page_content for doc in docs
    )

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

parallel = {
        "context": retriever | RunnableLambda(formatDoc),
        "question": RunnablePassthrough()
    }


chain = parallel | prompt | llm



In [ ]:
response = chain.invoke(
    "What is the sales report"
)

print(response.content)